In [21]:
import re
import urllib.request
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"

BOOKS = {
    "frankenstein": "https://www.gutenberg.org/cache/epub/84/pg84.txt",
    "timemachine": "https://www.gutenberg.org/cache/epub/35/pg35.txt",
}

def load_book(name, save_dir="data"):
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    path = Path(save_dir) / f"{name}.txt"
    if not path.exists():
        req = urllib.request.Request(BOOKS[name], headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp:
            raw = resp.read().decode("utf-8-sig")
        start = re.search(r"\*\*\* START OF THE PROJECT GUTENBERG EBOOK .*? \*\*\*", raw)
        end = re.search(r"\*\*\* END OF THE PROJECT GUTENBERG EBOOK .*? \*\*\*", raw)
        clean = raw[start.end() if start else 0 : end.start() if end else len(raw)].strip()
        path.write_text(clean, encoding="utf-8")
    return path.read_text(encoding="utf-8")

text = load_book("frankenstein") + "\n\n" + load_book("timemachine")

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in stoi.items()}

def encode(s):
    return [stoi[c] for c in s if c in stoi]

def decode(ids):
    return "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

config = {
    "vocab_size": vocab_size,
    "block_size": 64,
    "n_embd": 96,
    "n_heads": 4,
    "n_layers": 4,
    "dropout": 0.1,
}
batch_size = 32

def get_batch(split):
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - config["block_size"], (batch_size,))
    x = torch.stack([src[i : i + config["block_size"]] for i in ix])
    y = torch.stack([src[i + 1 : i + config["block_size"] + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print(f"Device: {device} | Corpus chars: {len(data):,} | Vocab size: {vocab_size}")
print(f"Batch X shape: {tuple(xb.shape)} | Batch Y shape: {tuple(yb.shape)}")

Device: cpu | Corpus chars: 599,036 | Vocab size: 89
Batch X shape: (32, 64) | Batch Y shape: (32, 64)


In [22]:
class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = self.dropout(F.softmax(wei, dim=-1))
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_heads, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_heads
        self.heads = nn.ModuleList(
            [Head(n_embd, head_size, block_size, dropout) for _ in range(n_heads)]
        )
        self.proj = nn.Linear(n_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_heads, block_size, dropout):
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_heads, block_size, dropout)
        self.ff = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.block_size = cfg["block_size"]
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["n_embd"])
        self.position_embedding = nn.Embedding(cfg["block_size"], cfg["n_embd"])
        self.blocks = nn.Sequential(
            *[
                Block(cfg["n_embd"], cfg["n_heads"], cfg["block_size"], cfg["dropout"])
                for _ in range(cfg["n_layers"])
            ]
        )
        self.ln_f = nn.LayerNorm(cfg["n_embd"])
        self.lm_head = nn.Linear(cfg["n_embd"], cfg["vocab_size"])

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(pos)
        x = self.ln_f(self.blocks(x))
        logits = self.lm_head(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

model = GPT(config).to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {param_count:,}")

Model parameters: 469,721


In [ ]:
max_iters = 3000
eval_interval = 500
eval_iters = 50
learning_rate = 3e-3

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
history = []

for step in range(max_iters + 1):
    if step % eval_interval == 0:
        losses = estimate_loss()
        history.append((step, losses["train"], losses["val"]))
        print(f"Step {step:4d} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")
    if step == max_iters:
        break
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

checkpoint = {
    "model_state": model.state_dict(),
    "config": config,
    "stoi": stoi,
    "itos": itos,
}
torch.save(checkpoint, "gpt.pt")
print("Saved self-contained checkpoint to gpt.pt")

In [ ]:
class MiniGPT:
    def __init__(self, weights="gpt.pt", device="cpu"):
        self.device = device
        ckpt = torch.load(weights, map_location=device)
        self.config = ckpt["config"]
        self.stoi = ckpt["stoi"]
        self.itos = ckpt["itos"]
        self.model = GPT(self.config).to(device)
        self.model.load_state_dict(ckpt["model_state"])
        self.model.eval()

    def _encode(self, s):
        return [self.stoi[c] for c in s if c in self.stoi]

    def _decode(self, ids):
        return "".join(self.itos[i] for i in ids)

    @torch.no_grad()
    def complete(self, prompt="\n", max_new_tokens=250, temperature=0.8, top_k=20, top_p=None):
        tokens = self._encode(prompt) or [0]
        idx = torch.tensor([tokens], dtype=torch.long, device=self.device)
        block_size = self.config["block_size"]

        for _ in range(max_new_tokens):
            logits, _ = self.model(idx[:, -block_size:])
            logits = logits[:, -1, :] / max(temperature, 1e-5)

            if top_k is not None:
                k = min(top_k, logits.size(-1))
                kth = torch.topk(logits, k)[0][:, [-1]]
                logits = logits.masked_fill(logits < kth, float("-inf"))

            if top_p is not None:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[:, 1:] = sorted_indices_to_remove[:, :-1].clone()
                sorted_indices_to_remove[:, 0] = False
                indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
                logits = logits.masked_fill(indices_to_remove, float("-inf"))

            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)

        return self._decode(idx[0].tolist())

gpt = MiniGPT("gpt.pt", device=device)

for temp, k, p in [(0.5, 10, None), (0.8, 25, None), (1.0, None, 0.9)]:
    print(f"=== temperature={temp}, top_k={k}, top_p={p} ===")
    print(gpt.complete(prompt="The creature looked ", max_new_tokens=220, temperature=temp, top_k=k, top_p=p))
    print()